# 📈Retail Insights Lab  

# Objetivo General

### Explorar y comprender los patrones de ventas retail, la influencia de factores externos (clima, economía, festivos) y las características de las tiendas, para generar insights estratégicos que apoyen decisiones de marketing, inventario y expansión.

In [1]:
.venv\Scripts\activate  

SyntaxError: invalid syntax (3720863216.py, line 1)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import sqlite3
from IPython.display import display



ModuleNotFoundError: No module named 'pandas'

In [ ]:

# Conexión a base de datos SQLite
conn = sqlite3.connect("retail_sales.db")

In [ ]:
# Cargar CSV a DataFrame
sales = pd.read_csv("sales.csv")
stores = pd.read_csv("stores.csv")
features = pd.read_csv("features.csv")

# Descripción de columnas por tabla
## Tabla: sales
store_id → Identificador único de la tienda./
department → Departamento o categoría de producto./ 
date → Fecha de la venta semanal./ 
weekly_sales → Ventas totales de ese departamento en esa semana./ 
is_holiday → Indicador binario (1 = semana festiva, 0 = semana normal).

## Tabla: stores
store_id → Identificador único de la tienda./ 
store_type → Clasificación de la tienda (ej. A, B, C según tamaño o formato)./ 
store_size → Tamaño de la tienda en pies cuadrados o metros cuadrados./ 
region → Región geográfica donde se ubica la tienda.

## Tabla: features
store_id → Identificador de la tienda./ 
date → Fecha de la observación semanal./ 
temperature → Temperatura promedio de la región esa semana./ 
fuel_price → Precio del combustible en la región./ 
markdown_1 … markdown_5 → Rebajas/promociones aplicadas a diferentes categorías de productos./ 
cpi → Índice de precios al consumidor (inflación)./ 
unemployment → Tasa de desempleo en la región./ 
is_holiday → Indicador de semana festiva./ 
holiday_name → Nombre del festivo (ej. “New Year”)./ 
season → Estación del año (Winter, Summer, etc.).

In [ ]:
# Mostrar las tres tablas en una sola celda
display(sales.head())
display(stores.head())
display(features.head())

,store_id,department,date,weekly_sales,is_holiday
0,1,1,2022-01-01,119075.96,1
1,1,2,2022-01-01,119107.85,1
2,1,3,2022-01-01,84369.88,1
3,1,4,2022-01-01,88445.24,1
4,1,5,2022-01-01,65159.85,1


,store_id,store_type,store_size,region
0,1,A,213810,North
1,2,C,31639,East
2,3,B,102098,South
3,4,B,88289,North
4,5,A,218696,North


,store_id,date,temperature,fuel_price,markdown_1,markdown_2,markdown_3,markdown_4,markdown_5,cpi,unemployment,is_holiday,holiday_name,season
0,1,2022-01-01,97.57,4.83,10334.49,5905.86,5261.52,7098.43,876.08,203.52,3.32,1,New Year,Winter
1,1,2022-01-08,46.03,3.67,1356.75,2486.21,1427.01,983.27,2442.13,196.91,8.62,0,NaN,Winter
2,1,2022-01-15,25.96,5.46,3861.22,596.15,22.09,2854.11,3180.86,267.48,8.40,0,NaN,Winter
3,1,2022-01-22,25.92,3.58,579.35,2589.31,2493.19,1158.14,286.01,217.32,5.28,0,NaN,Winter
4,1,2022-01-29,78.37,4.41,4436.06,1416.64,478.38,2496.36,3423.53,247.35,8.40,0,NaN,Winter


# Limpieza de datos (ETL)

In [ ]:
sales['date']= pd.to_datetime(sales['date'], format='%Y-%m-%d')
sales.info()


<class 'pandas.DataFrame'>
RangeIndex: 156000 entries, 0 to 155999
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   store_id      156000 non-null  int64         
 1   department    156000 non-null  int64         
 2   date          156000 non-null  datetime64[us]
 3   weekly_sales  156000 non-null  float64       
 4   is_holiday    156000 non-null  int64         
dtypes: datetime64[us](1), float64(1), int64(3)
memory usage: 6.0 MB


In [ ]:
features['date']= pd.to_datetime(features['date'], format= '%Y-%m-%d')
features.info()


<class 'pandas.DataFrame'>
RangeIndex: 7800 entries, 0 to 7799
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   store_id      7800 non-null   int64         
 1   date          7800 non-null   datetime64[us]
 2   temperature   7800 non-null   float64       
 3   fuel_price    7800 non-null   float64       
 4   markdown_1    7800 non-null   float64       
 5   markdown_2    7800 non-null   float64       
 6   markdown_3    7800 non-null   float64       
 7   markdown_4    7800 non-null   float64       
 8   markdown_5    7800 non-null   float64       
 9   cpi           7800 non-null   float64       
 10  unemployment  7800 non-null   float64       
 11  is_holiday    7800 non-null   int64         
 12  holiday_name  900 non-null    str           
 13  season        7800 non-null   str           
dtypes: datetime64[us](1), float64(9), int64(2), str(2)
memory usage: 853.3 KB


In [ ]:
stores.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   store_id    50 non-null     int64
 1   store_type  50 non-null     str  
 2   store_size  50 non-null     int64
 3   region      50 non-null     str  
dtypes: int64(2), str(2)
memory usage: 1.7 KB


Se detectó que en las tablas features y sales la columna que pertenece a "fecha" tenía un tipo de formato que podría impedir un análsis correcto, por lo que se cambió a datetime para una investigación fluida 


In [ ]:

# Guardar en SQL
sales.to_sql("ventas", conn, if_exists="replace", index=False)
stores.to_sql("tiendas", conn, if_exists="replace", index=False)
features.to_sql("features", conn, if_exists="replace", index=False)

NameError: name 'sales' is not defined

In [ ]:
# Visualización con Seaborn
sns.lineplot(data=sales, x="date", y="weekly_sales")
plt.title("Ventas semanales")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Ejemplo de consulta SQL
query = """
SELECT strftime('%Y-%m', date) AS mes, SUM(weekly_sales) AS ventas_mensuales
FROM ventas
GROUP BY mes
ORDER BY mes;
"""
df_sql = pd.read_sql(query, conn)

In [ ]:
python retail_analysis.py
